# ViHBC 运行 NicheCompass + TMCN 完整 notebook

这个 notebook 是给你直接在 `TuMeNiche/src/my/ViHBC_run_nichecompass.ipynb` 里跑模型用的。

## 你只需要优先改 4 个地方
1. `DATA_PATH`：你的输入 h5ad 路径。
2. `QUADRUPLET_CSV`：你的四元组 CSV 路径。
3. `OUTPUT_DIR`：输出目录。
4. `PATHOLOGY_KEY`：如果病理标签列不是 `annot_type`，就改成你自己的列名。

其余代码先不要动。


In [ ]:
# ===== Cell 1: 固定 GPU（必须最先运行） =====
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
print("当前 notebook 只会看到 1 块 GPU，并且对应物理 GPU 0")


In [ ]:
# ===== Cell 2: 导入依赖 + 确认导入的是你本地源码 =====
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch

warnings.filterwarnings("ignore")

REPO_ROOT = "/home/zhangjunyi/xiangmu/nichecompass-main"
LOCAL_SRC = f"{REPO_ROOT}/src"

if LOCAL_SRC not in sys.path:
    sys.path.insert(0, LOCAL_SRC)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import nichecompass as nc

from analysis.data_analysis.tmcn_cluster_pipeline import (
    DEFAULT_PATHWAY_GENE_SETS,
    assign_cluster_labels,
    assign_spot_level_flags,
    compute_axis_scores,
    compute_pathology_evaluation,
    ensure_latent_clusters,
    ensure_spatial_graph,
    load_axis_definitions,
    refine_to_connected_regions,
)

print("nichecompass loaded from:")
print(nc.__file__)
print("torch.cuda.is_available() =", torch.cuda.is_available())
print("torch.cuda.device_count() =", torch.cuda.device_count())
if torch.cuda.is_available():
    print("当前 GPU =", torch.cuda.get_device_name(0))

assert nc.__file__.startswith(f"{REPO_ROOT}/src"), "当前导入的不是你本地修改后的 NicheCompass，请先重启 kernel 再运行。"


## 下面这个单元格是你最需要修改的位置

如果你是小白，**先只改这一格**：
- 数据路径
- 四元组路径
- 输出路径
- 病理标签列名
- 训练 epoch / cluster 分辨率

改完以后按顺序从上往下运行。


In [ ]:
# ===== Cell 3: 配置区（你重点修改这里） =====
DATA_PATH = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_integrated.h5ad"
QUADRUPLET_CSV = "/home/zhangjunyi/xiangmu/nichecompass-main/data/pre_data/siyuanzu/my_metabolite_network_simplify.csv"
OUTPUT_DIR = "/home/zhangjunyi/xiangmu/nichecompass-main/artifacts/ViHBC_TMCN_run"
PATHOLOGY_KEY = "annot_type"   # 如果你的病理标签列名不是 annot_type，就改这里

SPATIAL_KEY = "spatial"
SPATIAL_CONNECTIVITIES_KEY = "spatial_connectivities"
LATENT_KEY = "tmcn_latent"
CLUSTER_KEY = "TMCN_Clusters"

# 训练参数：第一次建议先用小一点，跑通后再调大
N_TOP_HVG = 2000
N_SPATIAL_NEIGHBORS = 8
CLUSTER_RESOLUTION = 0.60
SPOT_HIGH_QUANTILE = 0.80
MIN_REGION_SIZE = 20

HEALTHY_MAX_HIGH_RISK_FRACTION = 0.20
DOMINANT_AXIS_MIN_FRACTION = 0.30
DOMINANT_AXIS_MIN_MARGIN = 0.15
MIXED_AXIS_MIN_FRACTION = 0.20
MIXED_SPOT_MIN_FRACTION = 0.25
MIXED_SPOT_MIN_SHARE = 0.25
QUIESCENT_MIN_FRACTION = 0.50
MERGE_QUIESCENT_INTO_HEALTHY = True

N_EPOCHS = 100
N_EPOCHS_ALL_GPS = 20
EDGE_BATCH_SIZE = 256
NODE_BATCH_SIZE = 512
USE_CUDA_IF_AVAILABLE = True

# 如果你想手动修改通路 marker，就复制 DEFAULT_PATHWAY_GENE_SETS 后在这里改
PATHWAY_GENE_SETS = DEFAULT_PATHWAY_GENE_SETS.copy()

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR =", OUTPUT_DIR)


In [ ]:
# ===== Cell 4: 读取四元组 =====
axes = load_axis_definitions(QUADRUPLET_CSV)
axis_names = [axis.axis_name for axis in axes]

df_axes = pd.DataFrame([
    {
        "axis_name": axis.axis_name,
        "source_pathways": ", ".join(axis.pathway_names),
        "source_genes": ", ".join(axis.source_genes),
        "target_genes": ", ".join(axis.target_genes),
        "meaning": axis.meaning,
    }
    for axis in axes
])

print("共读取四元组条数：", len(axes))
display(df_axes)


In [ ]:
# ===== Cell 5: Notebook 内部辅助函数 =====
def split_items(x):
    return [i.strip() for i in str(x).split(",") if i.strip()]

def get_all_required_genes(axes, pathway_gene_sets):
    genes = set()
    for axis in axes:
        for p in axis.pathway_names:
            genes.update(pathway_gene_sets.get(p, []))
        genes.update(axis.source_genes)
        genes.update(axis.target_genes)
    return genes

def prepare_training_adata(adata_full, axes, pathway_gene_sets, n_top_hvg=2000):
    adata_tmp = adata_full.copy()

    if "counts" in adata_tmp.layers:
        adata_tmp.X = adata_tmp.layers["counts"].copy()

    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)
    sc.pp.highly_variable_genes(adata_tmp, n_top_genes=n_top_hvg, flavor="seurat_v3")

    hvg_genes = set(adata_tmp.var_names[adata_tmp.var["highly_variable"]].tolist())
    required_genes = get_all_required_genes(axes, pathway_gene_sets)
    required_genes = set(g for g in required_genes if g in adata_full.var_names)

    keep_genes = sorted(hvg_genes | required_genes)
    adata_model = adata_full[:, keep_genes].copy()
    return adata_model, keep_genes

def build_axis_gp_dict(axes, adata):
    gp_dict = {}
    valid_genes = set(adata.var_names)

    for axis in axes:
        src = [g for g in axis.source_genes if g in valid_genes]
        tgt = [g for g in axis.target_genes if g in valid_genes]

        if len(src) == 0 and len(tgt) == 0:
            continue

        gp_dict[axis.axis_name] = {
            "sources": src,
            "targets": tgt,
            "sources_categories": ["enzyme"] * len(src),
            "targets_categories": ["receptor"] * len(tgt),
        }

    return gp_dict

def build_tmcn_weighted_graph(adata, axis_names, base_key="spatial_connectivities", key_added="tmcn_connectivities"):
    base_graph = adata.obsp[base_key].tocsr().copy()
    rows, cols = base_graph.nonzero()

    weights = np.zeros(len(rows), dtype=float)
    for axis_name in axis_names:
        sender = adata.obs[f"{axis_name}__sender_score"].to_numpy()
        receiver = adata.obs[f"{axis_name}__receiver_score"].to_numpy()

        forward_score = sender[rows] * receiver[cols]
        reverse_score = sender[cols] * receiver[rows]
        axis_edge_score = np.maximum(forward_score, reverse_score)
        weights += axis_edge_score

    if len(axis_names) > 0:
        weights = weights / len(axis_names)

    weights = np.asarray(weights, dtype=float)
    if np.allclose(weights.max(), 0):
        weights = np.ones_like(weights)
    else:
        weights = weights / (weights.max() + 1e-8)

    weighted_graph = sp.csr_matrix((weights, (rows, cols)), shape=base_graph.shape)
    weighted_graph = weighted_graph.maximum(weighted_graph.T)
    adata.obsp[key_added] = weighted_graph
    return adata

def build_node_feature_keys(axis_names):
    keys = []
    for axis_name in axis_names:
        keys.extend([
            f"{axis_name}__pathway_score",
            f"{axis_name}__enzyme_score",
            f"{axis_name}__receptor_score",
            f"{axis_name}__sender_score",
            f"{axis_name}__receiver_score",
            f"{axis_name}__coupling_score",
        ])
    return keys


In [ ]:
# ===== Cell 6: 读取数据 + 计算四元组连续分数 + 构建训练对象 =====
adata_full = sc.read_h5ad(DATA_PATH)
print(adata_full)

if "counts" not in adata_full.layers:
    adata_full.layers["counts"] = adata_full.X.copy()

ensure_spatial_graph(
    adata=adata_full,
    spatial_key=SPATIAL_KEY,
    spatial_connectivities_key=SPATIAL_CONNECTIVITIES_KEY,
    n_neighbors=N_SPATIAL_NEIGHBORS,
)

axis_names = compute_axis_scores(adata_full, axes, PATHWAY_GENE_SETS)
assign_spot_level_flags(
    adata_full,
    axis_names,
    high_quantile=SPOT_HIGH_QUANTILE,
    mixed_spot_min_share=MIXED_SPOT_MIN_SHARE,
)

score_cols = [c for c in adata_full.obs.columns if c.endswith("_score") or c.endswith("_spot")]
print("打分列数量：", len(score_cols))
display(adata_full.obs[score_cols].head())

adata_model, keep_genes = prepare_training_adata(
    adata_full,
    axes,
    PATHWAY_GENE_SETS,
    n_top_hvg=N_TOP_HVG,
)
print(adata_model)
print("训练基因数：", len(keep_genes))

gp_dict = build_axis_gp_dict(axes, adata_model)
print("自定义 GP 数量：", len(gp_dict))

nc.utils.add_gps_from_gp_dict_to_adata(gp_dict=gp_dict, adata=adata_model)

adata_model = build_tmcn_weighted_graph(
    adata_model,
    axis_names=axis_names,
    base_key=SPATIAL_CONNECTIVITIES_KEY,
    key_added="tmcn_connectivities",
)

print("tmcn_connectivities 构建完成：", adata_model.obsp["tmcn_connectivities"])


In [ ]:
# ===== Cell 7: 初始化并训练 NicheCompass =====
node_feature_keys = build_node_feature_keys(axis_names)
print("node_feature_keys 数量：", len(node_feature_keys))

model = nc.models.NicheCompass(
    adata=adata_model,
    counts_key="counts" if "counts" in adata_model.layers else None,
    adj_key="tmcn_connectivities",
    gp_names_key="nichecompass_gp_names",
    active_gp_names_key="nichecompass_active_gp_names",
    gp_targets_mask_key="nichecompass_gp_targets",
    gp_targets_categories_mask_key="nichecompass_gp_targets_categories",
    gp_sources_mask_key="nichecompass_gp_sources",
    gp_sources_categories_mask_key="nichecompass_gp_sources_categories",
    latent_key=LATENT_KEY,
    conv_layer_encoder="gcnconv",
    node_feature_keys=node_feature_keys,
    n_addon_gp=0,
)

print("开始训练...")
model.train(
    n_epochs=N_EPOCHS,
    n_epochs_all_gps=N_EPOCHS_ALL_GPS,
    edge_batch_size=EDGE_BATCH_SIZE,
    node_batch_size=NODE_BATCH_SIZE,
    use_cuda_if_available=USE_CUDA_IF_AVAILABLE,
)
print("训练完成。")


In [ ]:
# ===== Cell 8: 取 latent + 做 cluster 级生态位命名 =====
adata_model.obsm[LATENT_KEY] = model.get_latent_representation(
    adata=adata_model,
    counts_key="counts" if "counts" in adata_model.layers else None,
    adj_key="tmcn_connectivities",
    node_batch_size=NODE_BATCH_SIZE,
)

cluster_key_used = ensure_latent_clusters(
    adata=adata_model,
    latent_key=LATENT_KEY,
    cluster_key=None,
    neighbors_key="tmcn_latent_neighbors",
    cluster_resolution=CLUSTER_RESOLUTION,
)

adata_model.obs[CLUSTER_KEY] = adata_model.obs[cluster_key_used].astype(str)

cluster_summary = assign_cluster_labels(
    adata=adata_model,
    cluster_key=CLUSTER_KEY,
    axis_names=axis_names,
    healthy_max_high_risk_fraction=HEALTHY_MAX_HIGH_RISK_FRACTION,
    dominant_axis_min_fraction=DOMINANT_AXIS_MIN_FRACTION,
    mixed_axis_min_fraction=MIXED_AXIS_MIN_FRACTION,
    dominant_axis_min_margin=DOMINANT_AXIS_MIN_MARGIN,
    mixed_spot_min_fraction=MIXED_SPOT_MIN_FRACTION,
    quiescent_min_fraction=QUIESCENT_MIN_FRACTION,
    merge_quiescent_into_healthy=MERGE_QUIESCENT_INTO_HEALTHY,
)

refine_to_connected_regions(
    adata=adata_model,
    cluster_key=CLUSTER_KEY,
    spatial_connectivities_key=SPATIAL_CONNECTIVITIES_KEY,
    min_region_size=MIN_REGION_SIZE,
)

cluster_summary.to_csv(f"{OUTPUT_DIR}/tmcn_cluster_summary.csv", index=False)
display(cluster_summary.head(20))


In [ ]:
# ===== Cell 9: 用 pathology label 做后验评估（只评估，不训练） =====
if PATHOLOGY_KEY in adata_model.obs.columns:
    compute_pathology_evaluation(adata_model, PATHOLOGY_KEY, Path(OUTPUT_DIR))
    print("已输出 pathology 对照结果到：", OUTPUT_DIR)
else:
    print(f"警告：adata_model.obs 中没有列 {PATHOLOGY_KEY}，跳过 pathology 后验评估。")


In [ ]:
# ===== Cell 10: 空间可视化 =====
plt.rcParams["figure.figsize"] = (9, 9)

sc.pl.spatial(
    adata_model,
    color="tmcn_region_base_label",
    spot_size=170,
    title="Final TMCN region labels",
    show=True,
)

sc.pl.spatial(
    adata_model,
    color="tmcn_cluster_label",
    spot_size=170,
    title="Cluster-level TMCN labels",
    show=True,
)

if PATHOLOGY_KEY in adata_model.obs.columns:
    sc.pl.spatial(
        adata_model,
        color=PATHOLOGY_KEY,
        spot_size=170,
        title="Pathology labels (post-hoc only)",
        show=True,
    )


In [ ]:
# ===== Cell 11: 保存结果 =====
result_h5ad = f"{OUTPUT_DIR}/ViHBC_TMCN_result.h5ad"
adata_model.write_h5ad(result_h5ad)

spot_cols = [
    CLUSTER_KEY,
    "tmcn_dominant_axis",
    "tmcn_high_risk_spot",
    "tmcn_cluster_label",
    "tmcn_region_label",
    "tmcn_region_base_label",
]
for axis_name in axis_names:
    spot_cols.extend([
        f"{axis_name}__pathway_score",
        f"{axis_name}__enzyme_score",
        f"{axis_name}__receptor_score",
        f"{axis_name}__sender_score",
        f"{axis_name}__receiver_score",
        f"{axis_name}__coupling_score",
        f"{axis_name}__active_spot",
    ])
if PATHOLOGY_KEY in adata_model.obs.columns:
    spot_cols.append(PATHOLOGY_KEY)

adata_model.obs[spot_cols].to_csv(f"{OUTPUT_DIR}/ViHBC_TMCN_spot_results.csv")

model.save(
    dir_path=f"{OUTPUT_DIR}/trained_model",
    overwrite=True,
    save_adata=True,
    adata_file_name="adata.h5ad",
)

print("结果 h5ad：", result_h5ad)
print("spot 结果表：", f"{OUTPUT_DIR}/ViHBC_TMCN_spot_results.csv")
print("模型目录：", f"{OUTPUT_DIR}/trained_model")


## 你后面调参时，只改这几个参数就够了

### 如果结果太碎
- 增大 `MIN_REGION_SIZE`
- 提高 `SPOT_HIGH_QUANTILE`
- 降低 `CLUSTER_RESOLUTION`

### 如果结果太保守，很多都被分到 Healthy
- 降低 `SPOT_HIGH_QUANTILE`
- 提高 `CLUSTER_RESOLUTION`

### 如果 Mixed niche 太少
- 降低 `MIXED_SPOT_MIN_FRACTION`
- 降低 `MIXED_SPOT_MIN_SHARE`
- 降低 `DOMINANT_AXIS_MIN_MARGIN`

### 如果 Quiescent niche 太少
- 降低 `QUIESCENT_MIN_FRACTION`
- 结合 `cluster_summary` 里的 `quiescent_fraction` / `Top1_Fraction` / `Top2_Fraction` 一起看

### 如果训练太慢
- 先把 `N_EPOCHS` 改成 30
- 把 `EDGE_BATCH_SIZE` / `NODE_BATCH_SIZE` 再调小一点

先跑通，再优化。
### 如果你希望 Quiescent 直接并到 Healthy
- 保持 `MERGE_QUIESCENT_INTO_HEALTHY = True`（默认）
- 这样低耦合区域会保留 `tmcn_is_quiescent_pattern` 诊断列，但最终标签输出为 `Healthy_niche`

